In [ ]:
# Pin the branch so a later push cannot silently change what a finished run
# was produced by. If the repo is private, add a token:
#   https://<PAT>@github.com/Rifat137710/Public.git
!rm -rf /kaggle/working/safesac-repo
!git clone -q --branch claude/thesis-q1-journal-path-komv2c \
    https://github.com/Rifat137710/Public.git /kaggle/working/safesac-repo
%cd /kaggle/working/safesac-repo
!git rev-parse --short HEAD


In [ ]:
!pip -q install -r requirements.txt 2>&1 | tail -3
import torch, pandapower, cvxpy, clarabel, gymnasium
print("torch", torch.__version__)
print("pandapower", pandapower.__version__, "(must be 3.2.0)")
print("cvxpy", cvxpy.__version__, "| gymnasium", gymnasium.__version__)


In [ ]:
!python -m pytest tests/test_powerflow.py -q 2>&1 | tail -4
!python -u scripts/transfer_study.py --seeds 0 --episodes 3 --eval-episodes 2 \
    --train-z 0.5 --deploy-z 0.5 6.0 --out-dir /kaggle/working/smoke \
    2>&1 | grep -v "UserWarning\|warnings.warn" | tail -12


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

!python -u scripts/transfer_study.py \
    --seeds 0 1 2 3 4 \
    --episodes 200 \
    --eval-episodes 20 \
    --alpha 0.003 \
    --train-z 0.5 \
    --deploy-z 0.5 4.0 6.0 8.0 10.0 12.0 \
    --load 0.40 \
    --evs 30 \
    --out-dir /kaggle/working/results/transfer \
    2>&1 | grep -v "UserWarning\|warnings.warn"


In [ ]:
import json, shutil
from pathlib import Path

d = json.loads(Path("/kaggle/working/results/transfer/transfer.json").read_text())
print("train Z", d["train_z_pct"], "| deploy", d["deploy_z_pct"],
      "| seeds", d["seeds"], "| fingerprint", d["train_fingerprint"])
print(f"{'Z%':>6}{'A raw':>18}{'B frozen':>18}{'C deploy':>18}")
for z, row in d["summary"].items():
    print(f"{z:>6}" + "".join(
        f"{row[a]['viol'][0]:>9.4f}/{row[a]['soc'][0]:.3f}"
        for a in ("A_raw", "B_frozen_proj", "C_deploy_proj")))

shutil.make_archive("/kaggle/working/transfer_results", "zip",
                    "/kaggle/working/results")
print("\ndownload /kaggle/working/transfer_results.zip and send it back")
